# AUROC Analysis

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from quickrules.data_handling import calculate_score, count_all_rules, count_all_attributes, bold
from pathlib import Path
from sklearn.metrics import roc_auc_score
import pandas as pd
import numpy as np

In [3]:
data_sets = [  # actual set for QR
    'australian',
     'bands',
     'bupa',
     'cleveland',
     'dermatology',
     'ecoli',
     'glass',
     'heart',
     'ionosphere',
     'pima',
     'sonar',  # 70 features!
     'spectfheart',
     'vehicle',
     'vowel',
     'wine',
     'winequality-red',
     'wisconsin',
     'yeast'
]

In [4]:
data_base = Path.cwd() / 'keel-data'
scores = {}
results_base = Path.cwd()

Scores for FRRI

In [5]:
scores['frri'] = calculate_score(
        data_folder=data_base,
        results_folder=results_base / 'lower-approx-new-impl-probas-test',
        include=data_sets,
        metric=roc_auc_score
    )

ValueError: could not convert string to float: '[0.05899 0.04775 0.10345 0.00000 0.00000 0.00000 0.01404 0.03252 0.06897'

Scores for QuickRules

Scores for MODLEM

## Table and statistical analysis

In [154]:
from quickrules.data_handling import get_dataset

fold = 0
fold_result_path = results_base / 'lower-approx-new-impl-probas-test' / 'ecoli' / f"fold{fold + 1}"
classes = list(np.load(str(fold_result_path / f"label_encoding_fold{fold + 1}.npy")))
_, y_test = get_dataset(data_base / 'ecoli-10-fold', f"{fold + 1}tst", get_datatypes=False)
y_test = np.array([classes.index(label) for label in y_test])
predictions = pd.read_csv(fold_result_path / f"fold{1}.dat",comment='@', header=None).values

In [158]:
classes

[' cp', ' im', ' imL', ' imS', ' imU', ' om', ' omL', ' pp']

In [155]:
predictions

array([[1.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000,
        0.00000],
       [1.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000,
        0.00000],
       [1.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000,
        0.00000],
       [0.00000, 0.00000, 0.00000, 0.00000, 1.00000, 0.00000, 0.00000,
        0.00000],
       [1.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000,
        0.00000],
       [0.02037, 0.67404, 0.00000, 0.00000, 0.30559, 0.00000, 0.00000,
        0.00000],
       [1.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000,
        0.00000],
       [1.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000,
        0.00000],
       [1.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000,
        0.00000],
       [1.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000,
        0.00000],
       [1.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000,
        0.00000],
       [1.00000, 0.00

In [156]:
np.sum(predictions, axis=1)

array([1.00000, 1.00000, 1.00000, 1.00000, 1.00000, 1.00000, 1.00000,
       1.00000, 1.00000, 1.00000, 1.00000, 1.00000, 1.00000, 1.00000,
       1.00000, 1.00000, 1.00000, 1.00000, 1.00000, 1.00000, 1.00000,
       1.00000, 1.00000, 1.00000, 1.00000, 1.00000, 1.00000, 1.00000,
       1.00000, 1.00000, 1.00000, 1.00000, 1.00000, 1.00000])

In [157]:
roc_auc_score(y_test, predictions, multi_class='ovo') # [:,1]

ValueError: Number of classes in y_true not equal to the number of columns in 'y_score'

Seeing where the array gets wonked up

In [148]:
x_train, y_train, t_train = get_dataset(data_base / 'ecoli-10-fold', f"{fold + 1}tra", get_datatypes=True)
x_test, y_test = get_dataset(data_base / 'ecoli-10-fold', f"{fold + 1}tst", get_datatypes=False)


In [149]:
classes = list(np.unique(np.append(y_train, y_test)))

In [150]:
y_train = np.array([classes.index(label) for label in y_train])
y_test = np.array([classes.index(label) for label in y_test])

In [151]:
from fuzzyroughrules.rule_induction_base import RuleGenerator
from fuzzyroughrules.approximations import LowerApproximation

model = RuleGenerator(LowerApproximation(), with_reducts=True)

In [152]:
model.fit(x_train, y_train, t_train)

In [153]:
probas = model.predict_proba(x_test, normalized=True)
probas

array([[1.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000,
        0.00000],
       [1.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000,
        0.00000],
       [1.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000,
        0.00000],
       [0.55592, 0.00000, 0.00000, 0.00000, 0.44408, 0.00000, 0.00000,
        0.00000],
       [1.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000,
        0.00000],
       [0.00000, 0.68806, 0.00000, 0.00000, 0.31194, 0.00000, 0.00000,
        0.00000],
       [1.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000,
        0.00000],
       [1.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000,
        0.00000],
       [1.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000,
        0.00000],
       [1.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000,
        0.00000],
       [1.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000,
        0.00000],
       [1.00000, 0.00

In [120]:
y_test.shape

(21,)

In [122]:
roc_auc_score(y_test, probas[:,1], multi_class='ovo') # [:,1]

0.9545454545454545

In [36]:
np.argmax(model.predict_proba(x_test, normalized=False).T, 1)

array([1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0])

In [37]:
np.argmax(model.predict_proba(x_test, normalized=False), 0)


array([1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0])

Figuring out the form of the tables we're using here

In [38]:
X_test = model.scaler.transform(x_test)

In [39]:
import fuzzyroughrules.fuzzy_operators as fo

In [41]:
# calculate credibility of each rule
credibility_predictions = []
for i in range(model.n_classes):
    credibility_predictions.append([])
for i in model.selected_indexes:
    credibility_predictions[model.y[i]].append(
        fo.lukasiewicz_t_norm(
            fo.general_triangular_relation(X_test, model.X_scaled[i], model.theta, model.reducts[i]),
            model.positive_region[i])
    )

In [50]:
len(model.selected_indexes)

18

In [51]:
len(credibility_predictions[0]) + len (credibility_predictions[1])

18

In [46]:
credibility_predictions  # now contains 1 list for each class
# each class list contains the matching the degree for each rule with each sample

[[array([0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000,
         0.00000, 0.00000, 0.00000, 0.31754, 0.00000, 0.00000, 0.00000,
         0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000]),
  array([0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000,
         0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.18036, 0.00000,
         0.00000, 0.00000, 0.00000, 0.00000, 0.02225, 0.00000, 0.00000]),
  array([0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000,
         0.00000, 0.05882, 0.00000, 0.00000, 0.00000, 0.00000, 0.17647,
         0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000]),
  array([0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000,
         0.00848, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.01354,
         0.05000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000]),
  array([0.00000, 0.00000, 0.09922, 0.00000, 0.00000, 0.00000, 0.00000,
         0.00000, 0.00000, 0.00000, 0.00000, 0.00000, 0.

In [52]:
# sum credibility for each class
cumulative_credibility = []
for i in range(model.n_classes):
    cumulative_credibility.append(np.max(np.array(credibility_predictions[i]), 0))

In [67]:
cumulative_credibility

[array([0.00000, 0.00000, 0.09922, 0.00000, 0.00000, 0.00000, 0.00000,
        0.06898, 0.05882, 0.00000, 0.31754, 0.02104, 0.18036, 0.17647,
        0.05000, 0.00000, 0.00000, 0.17956, 0.21134, 0.14873, 0.11896]),
 array([0.22339, 0.15703, 0.03307, 0.00000, 0.13689, 0.10025, 0.22047,
        0.29161, 0.06968, 0.04878, 0.00000, 0.00000, 0.00000, 0.01141,
        0.01090, 0.15126, 0.00000, 0.00000, 0.00000, 0.00000, 0.00000])]

In [56]:
nice_credibility = np.array(cumulative_credibility).T
nice_credibility

array([[0.00000, 0.22339],
       [0.00000, 0.15703],
       [0.09922, 0.03307],
       [0.00000, 0.00000],
       [0.00000, 0.13689],
       [0.00000, 0.10025],
       [0.00000, 0.22047],
       [0.06898, 0.29161],
       [0.05882, 0.06968],
       [0.00000, 0.04878],
       [0.31754, 0.00000],
       [0.02104, 0.00000],
       [0.18036, 0.00000],
       [0.17647, 0.01141],
       [0.05000, 0.01090],
       [0.00000, 0.15126],
       [0.00000, 0.00000],
       [0.17956, 0.00000],
       [0.21134, 0.00000],
       [0.14873, 0.00000],
       [0.11896, 0.00000]])

In [ ]:
total = sum(cumulative_credibility)
if total != 0:
    for index, val in enumerate(cumulative_credibility):
        cumulative_credibility[index] = val / total

return np.array(cumulative_credibility)

In [59]:
from sklearn.preprocessing import normalize

normalized_credibility = normalize(nice_credibility, norm='l1')

In [60]:
normalized_credibility

array([[0.00000, 1.00000],
       [0.00000, 1.00000],
       [0.75003, 0.24997],
       [0.00000, 0.00000],
       [0.00000, 1.00000],
       [0.00000, 1.00000],
       [0.00000, 1.00000],
       [0.19130, 0.80870],
       [0.45775, 0.54225],
       [0.00000, 1.00000],
       [1.00000, 0.00000],
       [1.00000, 0.00000],
       [1.00000, 0.00000],
       [0.93927, 0.06073],
       [0.82098, 0.17902],
       [0.00000, 1.00000],
       [0.00000, 0.00000],
       [1.00000, 0.00000],
       [1.00000, 0.00000],
       [1.00000, 0.00000],
       [1.00000, 0.00000]])

In [63]:
np.argmax(normalized_credibility, 1)

array([1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0])

In [73]:
test_array = np.zeros((X_test.shape[0], model.n_classes))

In [74]:
test_array

array([[0.00000, 0.00000],
       [0.00000, 0.00000],
       [0.00000, 0.00000],
       [0.00000, 0.00000],
       [0.00000, 0.00000],
       [0.00000, 0.00000],
       [0.00000, 0.00000],
       [0.00000, 0.00000],
       [0.00000, 0.00000],
       [0.00000, 0.00000],
       [0.00000, 0.00000],
       [0.00000, 0.00000],
       [0.00000, 0.00000],
       [0.00000, 0.00000],
       [0.00000, 0.00000],
       [0.00000, 0.00000],
       [0.00000, 0.00000],
       [0.00000, 0.00000],
       [0.00000, 0.00000],
       [0.00000, 0.00000],
       [0.00000, 0.00000]])

In [75]:
for i in range(model.n_classes):
    test_array[:,i]=np.max(np.array(credibility_predictions[i]), 0)

In [76]:
test_array

array([[0.00000, 0.22339],
       [0.00000, 0.15703],
       [0.09922, 0.03307],
       [0.00000, 0.00000],
       [0.00000, 0.13689],
       [0.00000, 0.10025],
       [0.00000, 0.22047],
       [0.06898, 0.29161],
       [0.05882, 0.06968],
       [0.00000, 0.04878],
       [0.31754, 0.00000],
       [0.02104, 0.00000],
       [0.18036, 0.00000],
       [0.17647, 0.01141],
       [0.05000, 0.01090],
       [0.00000, 0.15126],
       [0.00000, 0.00000],
       [0.17956, 0.00000],
       [0.21134, 0.00000],
       [0.14873, 0.00000],
       [0.11896, 0.00000]])

In [127]:
repr(list(probas[0,:]))

'[0.0, 1.0]'

In [139]:
file = Path.cwd() / 'test_save_f.txt'

with open(file, mode = 'w') as f:
    f.write('a first line \n')
    # probas.tofile(f, sep=", ")
    np.savetxt(f, model.predict(X_test), delimiter=",", fmt='%.5f') #  '%i'